NetworkX-Temporal
---

- Links:
[Documentation](https://networkx-temporal.readthedocs.io/en/stable/examples/basics.html) |
[PyPI project](https://pypi.org/p/networkx-temporal/) |
[GitHub repository](https://github.com/nelsonaloysio/networkx-temporal)

- Examples:
[Basic operations](networkx-temporal-01-basics.ipynb) |
[Convert and transform](networkx-temporal-02-convert.ipynb) |
[Algorithms and metrics](networkx-temporal-03-metrics.ipynb) |
[Community detection](networkx-temporal-04-community.ipynb) |
[GPU acceleration](networkx-temporal-05-gpu.ipynb)

In [ ]:
#!pip install -q 'networkx-temporal[ipynb]'   # Installs additional libraries used in this notebook.

In [ ]:
#!conda install -c conda-forge -c rapidsai -c nvidia cupy cuml cugraph pylibcugraph nx-cugraph  # Installs required GPU-based libraries.

___

# GPU acceleration

Some algorithms for temporal networks can be computationally intensive, especially for large-scale
networks. To address this, NetworkX-Temporal integrates some GPU-accelerated algorithm implementations using the
open-source libraries CuPy, cuGraph, and cuML, substantially their reducing computation time.
This guide illustrates some commomn usage examples.

Enabling GPU acceleration requires the installation of the following libraries for hardware with CUDA support:

- [CuPy](https://docs.cupy.dev/en/stable/install.html): A library for array computations on GPUs, similar to NumPy and SciPy;
- [cuGraph](https://docs.rapids.ai/api/cugraph/stable/install.html): Graph computations part of the RAPIDS suite for NVIDIA GPUs;
- [cuML](https://docs.rapids.ai/api/cuml/stable/install.html): Machine learning algorithms in the same ecosystem (NVIDIA GPUs).

Once installed, GPU acceleration in NetworkX-Temporal may be enabled by passing the
parameter `device="gpu"` in the relevant functions; or alternatively, by simply setting the
environmental variable `NX_CUGRAPH_AUTOCONFIG=1`, allowing zero-configuration GPU acceleration for
all supported algorithms:

In [ ]:
# Enables GPU acceleration by default for supported algorithms.
%env NX_CUGRAPH_AUTOCONFIG=1
%env CUDA_VISIBLE_DEVICES=4

In [ ]:
# Verify GPU acceleration is enabled by default.
import networkx_temporal as tx
tx.is_gpu_enabled

> For the time being, only algorithms relevant for community detection are implemented with support for GPU acceleration [1].

## Community detection

For a first example, we will use a synthetic temporal graph generated by the `example_sbm_graph` function, based on a stochastic block model (SBM).

In [ ]:
TG = tx.example_sbm_graph()
G = TG.to_static()

### Spectral clustering

The `spectral_clustering` function implements algorithms for clustering matrices derived from the graph `'laplacian'`, `'bethe_hessian'`, or `'modularity'` matrix.

In [ ]:
y_pred = tx.spectral_clustering(G, k=3, operator="laplacian")

Let's plot the ground truth and the predicted communities side by side to compare the results:

In [ ]:
import matplotlib.pyplot as plt
colors = plt.cm.tab10.colors

def plot_static_graph(G, y_pred):
    y_true = tx.get_node_attributes(G, "community", index=False)
    pos = tx.layout(G, layout="kamada_kawai")
    node_color = [
        [colors[m % len(colors)] for m in y]
        for y in [y_true, y_pred]
    ]
    # Draw ground truths (left) and spectral clustering results (right).
    return tx.draw(
        [G, G], figsize=(6, 3), pos=pos, node_size=70,
        title=["Ground Truths", "Clustering Results"],
        temporal_node_color=node_color)

plot_static_graph(G, y_pred)

Instead of running the algorithm on the aggregated graphs, we can also run it on the temporal graph
directly. The `spectral_clustering` function automatically handles the construction of a (supra-)graph
encoding temporal adjacencies, and the predicted communities are returned for each node in each time step.

In [ ]:
y_pred = tx.spectral_clustering(TG, k=3, operator="laplacian")

In [ ]:
def plot_temporal_graph(TG, y_pred):
    y_true = tx.get_node_attributes(TG, "community", index=False)
    pos = tx.layout(TG, layout="kamada_kawai")
    temporal_node_color = [
        [colors[m % len(colors)] for m in s]
        for y in [y_true, y_pred]
        for s in y
    ]
    # Draw ground truths (top) and spectral clustering results (bottom).
    return tx.draw(
        [*TG, *TG], figsize=(6, 3.5), nrows=2, ncols=3,
        pos=[*pos, *pos], node_size=50, title=False,
        suptitle=f"Ground Truths (top) vs. Clustering Results (bottom)",
        temporal_node_color=temporal_node_color)

plot_temporal_graph(TG, y_pred)

### Leiden communities

The `leiden_communities` function also supports GPU acceleration for greedy optimization with 'quality' functions such as modularity.

If a temporal graph is provided, a supra-graph encoding with inter-slice couplings is constructed, and the
multislice (temporal) modularity is optimized instead of the standard (static) modularity function:

In [ ]:
y_pred = tx.leiden_communities(TG)

In [ ]:
plot_temporal_graph(TG, y_pred)

> Unless needed, spectral clustering is often preferred over greedy optimization, especially when the number of communities is known.

## Compare running times

The CPU and GPU implementations of the Leiden algorithm run through different backends depending on the device and graph type.

- CPU: uses [leidenalg](https://github.com/vtraag/leidenalg) and [igraph](https://igraph.org/python/), implemented in Python and C++.
- GPU: uses [cuGraph](https://docs.rapids.ai/api/cugraph/stable/) for static graphs and a parallelized [CuPy](https://cupy.dev/) implementation for temporal graphs.

### Temporal multislice optimization

On a fixed budget (`max_iter=2`), the GPU implementations for large-scale graphs are significantly faster than on CPU, especially in the temporal case.

In [ ]:
TG = tx.pubmed_graph()
print(TG)

In [ ]:
%timeit y = tx.leiden_communities(TG, max_iter=2, device="cpu")
# 52.2 s ± 1.42 s per loop (mean ± std. dev. of 7 runs, 1 loop each)

In [ ]:
%timeit y = tx.leiden_communities(TG, max_iter=2, device="gpu")
# 1.36 s ± 40.7 ms per loop (mean ± std. dev. of 20 runs, 1 loop each)

When efficiency is a topmost concern, it is possible to disable the refinement phase of Leiden algorithm by setting `refine=False`,
yielding a parallelized Louvain-like multislice modularity optimization that is ~20% faster, but does not guarantee internally connected communities.

In [ ]:
%timeit y = tx.leiden_communities(TG, max_iter=2, device="gpu", refine=False)
# 1.23 s ± 5.89 ms per loop (mean ± std. dev. of 20 runs, 1 loop each)

### Static modularity optimization

On static graphs, the GPU advantage emerges only at scale, with the CPU implementation being faster for smaller graphs due to kernel launch overheads [1].

In [ ]:
G = TG.to_static()
print(G)

In [ ]:
%timeit y = tx.leiden_communities(G, max_iter=2, device="cpu")
# 1.11 s ± 25.9 ms per loop (mean ± std. dev. of 20 runs, 1 loop each)

In [ ]:
%timeit y = tx.leiden_communities(G, max_iter=2, device="gpu")
# 74.9 ms ± 15.5 ms per loop (mean ± std. dev. of 20 runs, 1 loop each)

> The default number of iterations differ among CPU (`2`) and GPU (`500`) implementations. Parallelization and different backends may also affect the results.

### Static supra-graph optimization

Lastly, it is possible to optimize (static) modularity on a supra-graph representation of the temporal graph.
This approach is substantially faster than the temporal (multislice) implementation, and allows for multi-GPU execution by
loading the supra-graph into a distributed GPU cluster with [Dask-cuGraph](https://docs.rapids.ai/api/dask-cugraph/stable/).

In [ ]:
adj = tx.to_supra_adjacency_matrix(TG)
G_supra = tx.from_scipy(adj)
print(G_supra)

In [ ]:
%time y = tx.leiden_communities(G_supra, max_iter=2, device="gpu")
# 303 ms ± 60.8 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)

Note that this implementation, however, optimizes a different objective function than multislice (temporal) modularity,
as a global null model is instead employed for the entire supra-graph, rather than a null model for each time slice.

## Compare detection accuracy

Let's now compare the clustering results of the different algorithms on the same graph, using the ground truth communities as reference. We'll employ a fixed budget of `max_iter=100` for all implementations, both for static and multislice modularity optimization.

In [ ]:
max_iter = 100

# Multislice modularity optimization.
TG = tx.example_sbm_graph()
y_cpu_TG = tx.leiden_communities(TG, device="cpu", max_iter=max_iter)
y_gpu_TG = tx.leiden_communities(TG, device="gpu", max_iter=max_iter)
y_gpu_TG_norefine = tx.leiden_communities(TG, device="gpu", max_iter=max_iter, refine=False)

# Static modularity optimization.
G = TG.to_static()
y_cpu_G = tx.leiden_communities(G, device="cpu", max_iter=max_iter)
y_gpu_G = tx.leiden_communities(G, device="gpu", max_iter=max_iter)

# Build supra-graph connecting nodes across time slices (default interslice_weight=1.0).
adj, offsets = tx.to_supra_adjacency_matrix(TG, return_offsets=True)

# Supra-graph (static) modularity optimization.
G_supra = tx.from_scipy(adj)
y_gpu_G_supra = tx.leiden_communities(G_supra, device="gpu", max_iter=max_iter)
y_gpu_G_supra = [y_gpu_G_supra[offsets[t]:offsets[t] + len(TG[t])] for t in range(len(TG))]

In [ ]:
y_true_G = tx.get_node_attributes(G, "community", index=False)
pos = tx.layout(G, layout="kamada_kawai")

node_color = [
    [colors[m % len(colors)] for m in y]
    for y in [y_true_G, y_cpu_G, y_gpu_G]
]

tx.draw(
    [G, G, G], figsize=(6, 2.5), pos=pos, node_size=50,
    title=["CPU", "Ground Truth (Static)", "GPU"],
    temporal_node_color=node_color)


In [ ]:
y_true_TG = tx.get_node_attributes(TG, "community", index=False)
pos = tx.layout(TG, layout="kamada_kawai")

temporal_node_color = [
    [colors[m % len(colors)] for m in s]
    for y in [y_true_TG, y_cpu_TG, y_gpu_TG, y_gpu_TG_norefine, y_gpu_G_supra]
    for s in y
]

title = [
    "", "Ground Truths (Temporal)", "",
    "", "CPU", "",
    "", "GPU", "",
    "", "GPU (No Refine)", "",
    "", "GPU (Supra)", "",
]

tx.draw(
    [*TG, *TG, *TG, *TG, *TG], figsize=(6, 9.5), nrows=5, ncols=3,
    pos=[*pos, *pos, *pos, *pos, *pos], node_size=50, names=False,
    title=title,
    temporal_node_color=temporal_node_color)

On the synthetic SBM instance, the predicted communities are visually similar to the ground truth in all cases, with the exception of the static supra-graph optimization, which yields a different partitioning. This is expected, as the approach optimizes the static objective function under a global null model instead.

As a workaround, coupling nodes across all slices may yield a more consistent partitioning, although it inflates node degrees under the global null model; coupling all slices makes this inflation uniform across nodes, while chain coupling penalizes interior slices more than boundary ones. Dynamic (normalized) weights preserve this uniformity while still coupling nearby slices more strongly than distant ones:

In [ ]:
T, omega = len(TG), 1.0

# Reweight interslice couplings based on inverse distance weighting, normalized and symmetrized.
raw = {(i, j): 1 / abs(i - j) for i in range(T) for j in range(T) if i != j}
row = {i: sum(raw[(i, k)] for k in range(T) if k != i) for i in range(T)}
w = {(i, j): omega * v * 0.5 * (1 / row[i] + 1 / row[j]) for (i, j), v in raw.items()}

# Build supra-graph connecting nodes across 'all' time slices.
adj, offsets = tx.to_supra_adjacency_matrix(
    TG, interslice_method="all", interslice_weights=w, return_offsets=True)

# Supra-graph (static) modularity optimization.
G_supra_ = tx.from_scipy(adj)
y_gpu_G_supra_ = tx.leiden_communities(G_supra_, max_iter=max_iter, device="gpu")
y_gpu_G_supra_ = [y_gpu_G_supra_[offsets[t]:offsets[t] + len(TG[t])] for t in range(len(TG))]

print(f"Graph:\n{G}\n",
      f"Supra-Graph:\n{G_supra}\n",
      f"Supra-Graph (Coupling All Slices):\n{G_supra_}", sep="\n")

In [ ]:
temporal_node_color = [
    [colors[m % len(colors)] for m in s]
    for y in [y_true_TG, y_gpu_G_supra, y_gpu_G_supra_]
    for s in y
]

title = [
    "", "Ground Truths (Temporal)", "",
    "", "GPU (Supra)", "",
    "", "GPU (Supra, Coupling 'all' Slices, Reweighted)", "",
]

tx.draw(
    [*TG, *TG, *TG], figsize=(6, 6), nrows=3, ncols=3,
    pos=[*pos, *pos, *pos], node_size=50, names=False,
    title=title,
    temporal_node_color=temporal_node_color)

> Note that coupling all slice pairs adds $\mathcal{O}(N \times T^2)$ edges to the supra-graph, which is practical for few snapshots but grows quickly with $T$.

However, results are still suboptimal, with 4 communities detected instead of 3 (bottom row). Coupling only consecutive slices fragments the partition well beyond expectation (middle row), and while all-to-all coupling with inverse distance weighting reintroduces some consistency, the global null model continues to favor a different partitioning than the one obtained by the per-slice null model - serving only as a proxy for temporal community detection. Meanwhile, under the per-slice multislice null model, inter-slice couplings carry no null term and therefore do not inflate node degrees, so the same graph is partitioned into the expected three communities regardless of the coupling topology. 

This example highlights the advantage of multislice modularity optimization over the static supra-graph surrogate, which offers a faster but less accurate alternative. Its applicability may be restricted to temporal graphs with dynamic community assignments, where tracking their evolution over time is of interest and more principled approaches such as spectral clustering or CPU-based methods may be too slow to be practical.

___

[1] Passos, N.A.R.A.; Carlini, E.; Trani, S. (2026).
Accelerating Dynamic Graph Clustering on GPU Architectures with cuGraph.
The 6th workshop on Flexible Resource and Application Management on the Edge (FRAME);
co-located with Euro-Par'26.
Pisa, Italy, Aug. 24–28, 2026.
doi: [10.48550/arXiv.2608.03695](https://doi.org/10.48550/arXiv.2608.03695)